# Termodinámica Estadística Clásica: Curso Riguroso

---

## Tabla de Contenidos

1. **Fundamentos: De la Mecánica a la Estadística**
   - ¿Por qué necesitamos mecánica estadística?
   - Microestados y macroestados
2. **Espacio de Fases y Teorema de Liouville**
   - Espacio $\Gamma$ y evolución temporal
   - Densidad de probabilidad y ergodicidad
3. **Ensamble Microcanónico** ($N, V, E$ fijos)
   - Entropía de Boltzmann: $S = k_B \ln \Omega$
   - Paradoja de Gibbs y factor $N!$
4. **Ensamble Canónico** ($N, V, T$ fijos)
   - Función de partición $Z$
   - Energía libre de Helmholtz
   - Fluctuaciones de energía
5. **Ensamble Gran Canónico** ($\mu, V, T$ fijos)
   - Gran función de partición $\mathcal{Z}$
   - Potencial gran canónico
6. **Gas Ideal Clásico y Distribución de Maxwell-Boltzmann**
   - Derivación rigurosa
   - Distribuciones de velocidad y energía
7. **Termodinámica desde la Mecánica Estadística**
   - Leyes de la termodinámica emergentes
   - Potenciales termodinámicos
   - Relaciones de Maxwell
8. **Transiciones de Fase: Modelo de Ising 2D**
   - Campo medio y solución exacta
   - Simulación Monte Carlo (Metropolis)
9. **Teorema de Equipartición y Límites de la Física Clásica**
   - Catástrofe ultravioleta
   - Calores específicos y la necesidad de la cuántica
10. **Ejercicios Propuestos y Resueltos**

---

**Prerrequisitos:** Mecánica clásica (Lagrangiana/Hamiltoniana), cálculo multivariable, probabilidad básica.

**Enfoque:** Este notebook cubre **exclusivamente la mecánica estadística clásica**, estableciendo los fundamentos necesarios antes de abordar la mecánica cuántica estadística.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from scipy.special import factorial
import sympy as sp
from sympy import symbols, Function, diff, simplify, exp, log, sqrt, pi, oo
from sympy import Rational, integrate, latex, Eq, solve as sp_solve, Sum, Symbol
from IPython.display import display, Math, Markdown
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 100,
})

# Constantes físicas
kB = 1.380649e-23   # J/K  (Boltzmann)
NA = 6.02214076e23   # 1/mol (Avogadro)
R_gas = kB * NA      # J/(mol·K) (constante de gases)

print("✅ Librerías y constantes cargadas.")
print(f"   kB = {kB:.6e} J/K")
print(f"   NA = {NA:.6e} 1/mol")
print(f"   R  = {R_gas:.4f} J/(mol·K)")

---

## 1. Fundamentos: De la Mecánica a la Estadística

### 1.1 ¿Por qué necesitamos mecánica estadística?

Un mol de gas contiene $N_A \approx 6 \times 10^{23}$ partículas. Resolver las ecuaciones de Hamilton para cada una es imposible. La **mecánica estadística** resuelve este problema describiendo el sistema con **probabilidades** en lugar de trayectorias individuales.

### 1.2 Microestados y Macroestados

| Concepto | Definición | Ejemplo |
|----------|-----------|---------|
| **Microestado** | Estado microscópico completo: $(q_1,...,q_{3N}, p_1,...,p_{3N})$ | Posiciones y momentos de todas las partículas |
| **Macroestado** | Estado descrito por variables macroscópicas: $(E, V, N, T, P, S)$ | "Gas a 300K y 1 atm" |
| **Multiplicidad** $\Omega$ | Número de microestados compatibles con un macroestado | $\Omega(E, V, N)$ |

> **Postulado fundamental:** En equilibrio, todos los microestados accesibles con la misma energía son **igualmente probables** (principio de igual probabilidad a priori).

### 1.3 El puente: Entropía de Boltzmann

$$\boxed{S = k_B \ln \Omega(E, V, N)}$$

Esta es la ecuación que conecta el mundo microscópico (mecánica) con el macroscópico (termodinámica).

In [ ]:
# Demostración: Multiplicidad y la aparición de la distribución más probable
# Ejemplo: N monedas lanzadas → macroestado = número de caras

from math import comb

N_coins = 100
k_range = np.arange(0, N_coins + 1)
omega = np.array([float(comb(N_coins, k)) for k in k_range])
S_coins = np.log(omega)  # entropía (en unidades de kB)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Multiplicidad Ω(k)
ax1 = axes[0]
ax1.plot(k_range, omega, 'C0', linewidth=2)
ax1.fill_between(k_range, omega, alpha=0.2, color='C0')
ax1.set_xlabel("$k$ (número de caras)")
ax1.set_ylabel("$\\Omega(k) = \\binom{N}{k}$")
ax1.set_title(f"Multiplicidad para N={N_coins} monedas")
ax1.axvline(x=50, color='red', linestyle='--', label='$k = N/2$')
ax1.legend()

# Panel 2: Entropía S(k) = ln Ω(k)
ax2 = axes[1]
ax2.plot(k_range, S_coins, 'C1', linewidth=2)
ax2.set_xlabel("$k$ (número de caras)")
ax2.set_ylabel("$S/k_B = \\ln \\Omega$")
ax2.set_title("Entropía de Boltzmann")
ax2.axvline(x=50, color='red', linestyle='--', label='Máximo en $k=N/2$')
ax2.legend()

# Panel 3: Probabilidad P(k) = Ω(k)/2^N
prob = omega / 2**N_coins
ax3 = axes[2]
ax3.plot(k_range, prob, 'C2', linewidth=2)
ax3.fill_between(k_range, prob, alpha=0.2, color='C2')
ax3.set_xlabel("$k$ (número de caras)")
ax3.set_ylabel("$P(k)$")
ax3.set_title("Probabilidad (pico extremadamente agudo)")

# Marcar región ±σ
mu = N_coins / 2
sigma = np.sqrt(N_coins) / 2
ax3.axvspan(mu - sigma, mu + sigma, alpha=0.15, color='red', label=f'$\\mu \\pm \\sigma = {mu:.0f} \\pm {sigma:.1f}$')
ax3.legend()

plt.tight_layout()
plt.show()

print(f"📊 Para N = {N_coins}:")
print(f"   Ω_total = 2^{N_coins} ≈ {2**N_coins:.2e}")
print(f"   Ω_max = C({N_coins},{N_coins//2}) ≈ {comb(N_coins, N_coins//2):.2e}")
print(f"   P(k=50) = {prob[50]:.4f}")
print(f"   σ/μ = {sigma/mu:.3f} → Para N ~ 10²³, las fluctuaciones son ~10⁻¹²")

---

## 2. Espacio de Fases y Teorema de Liouville

### 2.1 El espacio $\Gamma$

Para $N$ partículas en 3D, el **espacio de fases** tiene $6N$ dimensiones:

$$\Gamma = \{(q_1, ..., q_{3N}, p_1, ..., p_{3N})\}$$

Cada punto en $\Gamma$ es un **microestado** del sistema. La evolución temporal es una curva en $\Gamma$ dictada por las ecuaciones de Hamilton.

### 2.2 Densidad en el espacio de fases

La **función de distribución** $\rho(\mathbf{q}, \mathbf{p}, t)$ da la probabilidad de encontrar al sistema cerca del punto $(\mathbf{q}, \mathbf{p})$:

$$\rho(\mathbf{q}, \mathbf{p}, t) \geq 0, \qquad \int \rho \, d\Gamma = 1$$

### 2.3 Teorema de Liouville

> La densidad en el espacio de fases es **constante a lo largo de las trayectorias hamiltonianas**:
>
> $$\boxed{\frac{d\rho}{dt} = \frac{\partial \rho}{\partial t} + \{\rho, H\} = 0}$$

donde $\{,\}$ son los **corchetes de Poisson**. Esto significa que el "fluido" del espacio de fases es **incompresible**.

### 2.4 Hipótesis Ergódica

> **Hipótesis ergódica:** El promedio temporal de una observable $A$ sobre una trayectoria larga es igual al promedio sobre el ensamble:
>
> $$\langle A \rangle_{\text{tiempo}} = \langle A \rangle_{\text{ensamble}} = \int A(\mathbf{q}, \mathbf{p}) \rho(\mathbf{q}, \mathbf{p}) \, d\Gamma$$

Esta hipótesis justifica el uso de ensambles estadísticos.

In [ ]:
# Demostración visual: Ergodicidad en un oscilador armónico 2D
# Un sistema integrable recorre la superficie de energía constante

def harmonic_2d(t, state, wx=1.0, wy=np.sqrt(2)):
    """Oscilador armónico 2D con frecuencias inconmensurables."""
    q1, p1, q2, p2 = state
    return [p1, -wx**2 * q1, p2, -wy**2 * q2]

# Frecuencias inconmensurables → ergódico en el toro de energía
t_erg = np.linspace(0, 200, 50000)
state0_erg = [1.0, 0.0, 0.5, 0.3]
sol_erg = solve_ivp(harmonic_2d, (0, 200), state0_erg, t_eval=t_erg, 
                     method='RK45', rtol=1e-12)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Trayectoria en espacio de configuración
ax1 = axes[0]
ax1.plot(sol_erg.y[0], sol_erg.y[2], linewidth=0.2, alpha=0.6, color='C0')
ax1.set_xlabel("$q_1$")
ax1.set_ylabel("$q_2$")
ax1.set_title("Espacio de configuración ($\\omega_1/\\omega_2$ irracional)\nLa trayectoria llena densamente el espacio")
ax1.set_aspect('equal')

# Panel 2: Histograma 2D → distribución uniforme sobre la superficie
ax2 = axes[1]
h = ax2.hist2d(sol_erg.y[0], sol_erg.y[2], bins=50, cmap='inferno', density=True)
plt.colorbar(h[3], ax=ax2, label='Densidad')
ax2.set_xlabel("$q_1$")
ax2.set_ylabel("$q_2$")
ax2.set_title("Histograma temporal → distribución de ensamble\n(Verificación de ergodicidad)")

# Panel 3: Promedio temporal de q1² converge al promedio de ensamble
q1_sq_cumavg = np.cumsum(sol_erg.y[0]**2) / np.arange(1, len(sol_erg.y[0]) + 1)
# Promedio de ensamble teórico: <q1²> = E1/ω1² donde E1 = ½(p1² + ω1²q1²)
E1 = 0.5 * (state0_erg[1]**2 + 1.0**2 * state0_erg[0]**2)
q1sq_ensemble = E1 / (1.0**2)

ax3 = axes[2]
ax3.plot(t_erg, q1_sq_cumavg, 'C0', linewidth=1, label='Promedio temporal acumulado')
ax3.axhline(y=q1sq_ensemble, color='red', linestyle='--', linewidth=2, 
            label=f'Promedio de ensamble = {q1sq_ensemble:.3f}')
ax3.set_xlabel("Tiempo")
ax3.set_ylabel("$\\langle q_1^2 \\rangle$")
ax3.set_title("Convergencia: promedio temporal → ensamble")
ax3.legend()
ax3.set_xlim(0, 200)

plt.tight_layout()
plt.show()

print("✅ Hipótesis ergódica verificada numéricamente:")
print(f"   ⟨q₁²⟩_tiempo = {q1_sq_cumavg[-1]:.4f}")
print(f"   ⟨q₁²⟩_ensamble = {q1sq_ensemble:.4f}")

---

## 3. Ensamble Microcanónico ($N, V, E$ fijos)

### 3.1 Definición

El ensamble microcanónico describe un **sistema aislado** con energía, volumen y número de partículas fijos. La distribución de probabilidad es:

$$\rho(\mathbf{q}, \mathbf{p}) = \frac{1}{\Omega(E)} \delta(H(\mathbf{q}, \mathbf{p}) - E)$$

donde $\Omega(E)$ es el **volumen de la cáscara de energía** en el espacio de fases:

$$\Omega(E, V, N) = \frac{1}{N! h^{3N}} \int \delta(H - E) \, d^{3N}q \, d^{3N}p$$

- El factor $h^{3N}$ discretiza el espacio de fases (anticipando la cuántica)
- El factor $N!$ corrige el **sobreconteo de Gibbs** (partículas idénticas)

### 3.2 Gas ideal: Cálculo de $\Omega$

Para $N$ partículas libres en una caja de volumen $V$, $H = \sum_{i=1}^{3N} \frac{p_i^2}{2m}$.

La integral sobre momentos da el volumen de una hiperesfera $3N$-dimensional de radio $\sqrt{2mE}$:

$$\Omega(E, V, N) = \frac{V^N}{N! h^{3N}} \cdot \frac{2\pi^{3N/2}}{\Gamma(3N/2)} (2mE)^{(3N-1)/2} \Delta E$$

### 3.3 Entropía de Sackur-Tetrode

Aplicando $S = k_B \ln \Omega$ y la fórmula de Stirling ($\ln N! \approx N\ln N - N$):

$$\boxed{S(E, V, N) = N k_B \left[\ln\left(\frac{V}{N}\left(\frac{4\pi m E}{3 N h^2}\right)^{3/2}\right) + \frac{5}{2}\right]}$$

Esta es la **fórmula de Sackur-Tetrode**, verificada experimentalmente.

In [ ]:
# Paradoja de Gibbs y el factor N!
# Sin N!, la entropía no sería extensiva

print("═" * 60)
print("  PARADOJA DE GIBBS Y EL FACTOR N!")
print("═" * 60)

display(Markdown(r"""
**Sin el factor $N!$**, la entropía de mezcla de dos gases idénticos sería:

$$\Delta S_{\text{sin } N!} = 2Nk_B \ln 2 > 0 \quad \text{(¡INCORRECTO!)}$$

Mezclar dos porciones del **mismo** gas no debe cambiar la entropía. El factor $1/N!$ corrige esto al tratar las partículas como **indistinguibles**.

**Implicación profunda:** La indistinguibilidad de partículas idénticas es un hecho fundamental que la mecánica clásica no puede explicar naturalmente — es un preludio a la mecánica cuántica.
"""))

# Visualización: S extensiva vs no extensiva
N_range = np.arange(1, 51)
V_per_particle = 1.0  # V/N constante

# Entropía "correcta" (extensiva): S ∝ N
S_correct = N_range * (np.log(V_per_particle) + 2.5)

# Entropía "incorrecta" (sin N!): S tiene término extra N·ln(N)
S_incorrect = N_range * np.log(N_range * V_per_particle) + 1.5 * N_range * np.log(N_range) + 2.5 * N_range

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.plot(N_range, S_correct / N_range, 'C0', linewidth=2.5, label='Con $N!$ (extensiva)')
ax1.plot(N_range, S_incorrect / N_range, 'C1--', linewidth=2.5, label='Sin $N!$ (no extensiva)')
ax1.set_xlabel("$N$ (número de partículas)")
ax1.set_ylabel("$S/N$ (entropía por partícula)")
ax1.set_title("Extensividad de la entropía")
ax1.legend()
ax1.set_ylim(0, 20)

# Panel 2: Aproximación de Stirling
ax2 = axes[1]
n_stirling = np.arange(1, 30)
ln_factorial_exact = np.array([np.sum(np.log(np.arange(1, n+1))) for n in n_stirling])
ln_factorial_stirling = n_stirling * np.log(n_stirling) - n_stirling
ln_factorial_better = n_stirling * np.log(n_stirling) - n_stirling + 0.5 * np.log(2 * np.pi * n_stirling)

ax2.plot(n_stirling, ln_factorial_exact, 'ko', markersize=5, label='$\\ln(N!)$ exacto')
ax2.plot(n_stirling, ln_factorial_stirling, 'C1--', linewidth=2, label='$N\\ln N - N$ (Stirling)')
ax2.plot(n_stirling, ln_factorial_better, 'C2-', linewidth=2, label='$N\\ln N - N + \\frac{1}{2}\\ln(2\\pi N)$')
ax2.set_xlabel("$N$")
ax2.set_ylabel("$\\ln(N!)$")
ax2.set_title("Aproximación de Stirling")
ax2.legend()

plt.tight_layout()
plt.show()

---

## 4. Ensamble Canónico ($N, V, T$ fijos)

### 4.1 El sistema en contacto con un baño térmico

Cuando un sistema intercambia energía con un **reservorio** a temperatura $T$, la probabilidad de encontrarlo en un microestado con energía $E_i$ es:

$$\boxed{P_i = \frac{e^{-\beta E_i}}{Z}, \qquad \beta = \frac{1}{k_B T}}$$

Este es el **factor de Boltzmann**. La distribución exponencial decreciente penaliza los estados de alta energía.

### 4.2 Función de Partición Canónica

$$\boxed{Z(N, V, T) = \sum_i e^{-\beta E_i} = \frac{1}{N! h^{3N}} \int e^{-\beta H(\mathbf{q}, \mathbf{p})} d^{3N}q \, d^{3N}p}$$

La función de partición $Z$ es el objeto central: **toda la termodinámica se deriva de ella**.

### 4.3 Conexión con la termodinámica

| Cantidad | Fórmula desde $Z$ |
|----------|-------------------|
| Energía libre de Helmholtz | $F = -k_B T \ln Z$ |
| Entropía | $S = -\frac{\partial F}{\partial T}\bigg\|_{V,N} = k_B \ln Z + k_B T \frac{\partial \ln Z}{\partial T}$ |
| Energía media | $\langle E \rangle = -\frac{\partial \ln Z}{\partial \beta}$ |
| Presión | $P = -\frac{\partial F}{\partial V}\bigg\|_{T,N} = k_B T \frac{\partial \ln Z}{\partial V}$ |

### 4.4 Fluctuaciones de energía

$$\sigma_E^2 = \langle E^2 \rangle - \langle E \rangle^2 = \frac{\partial^2 \ln Z}{\partial \beta^2} = k_B T^2 C_V$$

Para $N$ grande: $\sigma_E / \langle E \rangle \sim 1/\sqrt{N} \to 0$. Las **fluctuaciones son despreciables** macroscópicamente.

In [ ]:
# Ensamble Canónico: Distribución de Boltzmann y función de partición
# Ejemplo: Sistema de 2 niveles (modelo magnético simple)

print("═" * 60)
print("  ENSAMBLE CANÓNICO: Sistema de 2 niveles")
print("═" * 60)

# Energías: E = ±ε (spin up/down en campo magnético)
epsilon = 1.0  # unidades de energía

beta_range = np.linspace(0.01, 5, 500)  # β = 1/(kT)
T_range = 1.0 / beta_range

# Función de partición: Z = e^{-βε} + e^{+βε} = 2cosh(βε)
Z_2lvl = 2 * np.cosh(beta_range * epsilon)

# Energía media: <E> = -∂lnZ/∂β = -ε·tanh(βε)
E_avg = -epsilon * np.tanh(beta_range * epsilon)

# Capacidad calorífica: Cv = -kβ² ∂<E>/∂β
Cv = (beta_range * epsilon)**2 / np.cosh(beta_range * epsilon)**2

# Entropía: S/k = ln Z + β<E>
S_2lvl = np.log(Z_2lvl) + beta_range * E_avg

# Energía libre: F = -kT ln Z = -(1/β) ln Z
F_2lvl = -np.log(Z_2lvl) / beta_range

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Distribución de Boltzmann para varias T
ax1 = axes[0, 0]
temps = [0.2, 0.5, 1.0, 2.0, 5.0]
colors_t = plt.cm.coolwarm(np.linspace(0.1, 0.9, len(temps)))
E_levels = np.array([-epsilon, epsilon])
width = 0.12

for i, T_val in enumerate(temps):
    beta_val = 1.0 / T_val
    Z_val = 2 * np.cosh(beta_val * epsilon)
    probs = np.exp(-beta_val * E_levels) / Z_val
    ax1.bar(E_levels + (i - 2) * width, probs, width=width, color=colors_t[i], 
            edgecolor='black', label=f'T = {T_val}', alpha=0.8)

ax1.set_xlabel("Energía $E$")
ax1.set_ylabel("Probabilidad $P(E)$")
ax1.set_title("Distribución de Boltzmann: $P \\propto e^{-E/k_BT}$")
ax1.legend()
ax1.set_xticks([-1, 1])
ax1.set_xticklabels(['$-\\varepsilon$', '$+\\varepsilon$'])

# Panel 2: Energía media
ax2 = axes[0, 1]
ax2.plot(T_range, E_avg / epsilon, 'C0', linewidth=2.5)
ax2.set_xlabel("$k_B T / \\varepsilon$")
ax2.set_ylabel("$\\langle E \\rangle / \\varepsilon$")
ax2.set_title("Energía media vs temperatura")
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='$T \\to \\infty$: equiprobable')
ax2.axhline(y=-1, color='C1', linestyle='--', alpha=0.5, label='$T \\to 0$: estado base')
ax2.legend()
ax2.set_xlim(0, 5)

# Panel 3: Capacidad calorífica (pico de Schottky)
ax3 = axes[1, 0]
ax3.plot(T_range, Cv, 'C1', linewidth=2.5)
ax3.set_xlabel("$k_B T / \\varepsilon$")
ax3.set_ylabel("$C_V / k_B$")
ax3.set_title("Capacidad calorífica (anomalía de Schottky)")
T_peak = T_range[np.argmax(Cv)]
ax3.axvline(x=T_peak, color='red', linestyle='--', alpha=0.5, label=f'Pico en $T \\approx {T_peak:.2f}\\varepsilon/k_B$')
ax3.legend()
ax3.set_xlim(0, 5)

# Panel 4: Entropía
ax4 = axes[1, 1]
ax4.plot(T_range, S_2lvl, 'C2', linewidth=2.5)
ax4.axhline(y=np.log(2), color='red', linestyle='--', alpha=0.5, label=f'$S_{{max}} = \\ln 2 \\approx {np.log(2):.3f}$')
ax4.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
ax4.set_xlabel("$k_B T / \\varepsilon$")
ax4.set_ylabel("$S / k_B$")
ax4.set_title("Entropía: de 0 (orden) a $\\ln 2$ (desorden)")
ax4.legend()
ax4.set_xlim(0, 5)

plt.tight_layout()
plt.show()

print(f"\n📊 Resultados del sistema de 2 niveles:")
print(f"   T → 0: sistema en estado base, S → 0 (3ª ley)")
print(f"   T → ∞: estados equiprobables, S → ln(2) = {np.log(2):.4f}")
print(f"   Cv tiene pico (Schottky) en T ≈ {T_peak:.2f} ε/kB")

---

## 5. Ensamble Gran Canónico ($\mu, V, T$ fijos)

### 5.1 Sistema abierto

Cuando el sistema intercambia **energía y partículas** con un reservorio:

$$P(E_i, N) = \frac{e^{-\beta(E_i - \mu N)}}{\mathcal{Z}}$$

### 5.2 Gran función de partición

$$\boxed{\mathcal{Z}(\mu, V, T) = \sum_{N=0}^{\infty} e^{\beta \mu N} Z(N, V, T) = \sum_{N=0}^{\infty} z^N Z(N, V, T)}$$

donde $z = e^{\beta \mu}$ es la **fugacidad**.

### 5.3 Potencial gran canónico

$$\Phi = -k_B T \ln \mathcal{Z} = F - \mu N = -PV$$

| Cantidad | Fórmula |
|----------|---------|
| $\langle N \rangle$ | $z \frac{\partial \ln \mathcal{Z}}{\partial z}$ |
| $\langle E \rangle$ | $-\frac{\partial \ln \mathcal{Z}}{\partial \beta} + \mu \langle N \rangle$ |
| $P V$ | $k_B T \ln \mathcal{Z}$ |

### 5.4 Resumen de ensambles

| Ensamble | Variables fijas | Potencial | Función de partición |
|----------|----------------|-----------|---------------------|
| Microcanónico | $N, V, E$ | $S(E,V,N)$ | $\Omega(E)$ |
| Canónico | $N, V, T$ | $F(T,V,N) = -k_BT\ln Z$ | $Z = \sum e^{-\beta E_i}$ |
| Gran canónico | $\mu, V, T$ | $\Phi(T,V,\mu) = -k_BT\ln\mathcal{Z}$ | $\mathcal{Z} = \sum z^N Z_N$ |

> **Equivalencia de ensambles:** En el límite termodinámico ($N \to \infty$), los tres ensambles dan los mismos resultados para cantidades intensivas.

---

## 6. Gas Ideal Clásico y Distribución de Maxwell-Boltzmann

### 6.1 Función de partición del gas ideal

Para $N$ partículas no interactuantes de masa $m$ en volumen $V$:

$$Z = \frac{1}{N!}\left(\frac{V}{\lambda^3}\right)^N, \qquad \lambda = \sqrt{\frac{2\pi\hbar^2}{mk_BT}} \quad \text{(longitud de onda térmica de De Broglie)}$$

### 6.2 Ecuación de estado

De $F = -k_BT\ln Z$ y $P = -(\partial F/\partial V)_T$:

$$\boxed{PV = Nk_BT}$$

¡La ecuación del gas ideal emerge naturalmente de la mecánica estadística!

### 6.3 Distribución de Maxwell-Boltzmann

La probabilidad de que una partícula tenga velocidad entre $\mathbf{v}$ y $\mathbf{v} + d\mathbf{v}$:

$$f(\mathbf{v}) = \left(\frac{m}{2\pi k_BT}\right)^{3/2} \exp\left(-\frac{mv^2}{2k_BT}\right)$$

La distribución de **rapideces** (speeds):

$$\boxed{f(v) = 4\pi v^2 \left(\frac{m}{2\pi k_BT}\right)^{3/2} \exp\left(-\frac{mv^2}{2k_BT}\right)}$$

Velocidades características:
- **Más probable:** $v_p = \sqrt{2k_BT/m}$
- **Media:** $\langle v \rangle = \sqrt{8k_BT/(\pi m)}$
- **RMS:** $v_{rms} = \sqrt{3k_BT/m}$

In [ ]:
# Distribución de Maxwell-Boltzmann: Visualización completa

def maxwell_speed_dist(v, m, T):
    """Distribución de Maxwell-Boltzmann de rapideces."""
    return 4 * np.pi * v**2 * (m / (2 * np.pi * kB * T))**1.5 * np.exp(-m * v**2 / (2 * kB * T))

def maxwell_energy_dist(E, T):
    """Distribución de energías cinéticas."""
    return 2 * np.pi * (1 / (np.pi * kB * T))**1.5 * np.sqrt(E) * np.exp(-E / (kB * T))

# Masa del nitrógeno (N2)
m_N2 = 28.0e-3 / NA  # kg por molécula

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: f(v) para diferentes temperaturas
ax1 = axes[0, 0]
temperatures = [100, 300, 600, 1000, 2000]
colors_mb = plt.cm.hot(np.linspace(0.2, 0.8, len(temperatures)))

for T_val, color in zip(temperatures, colors_mb):
    v_max = 5 * np.sqrt(2 * kB * T_val / m_N2)
    v_arr = np.linspace(0, v_max, 1000)
    fv = maxwell_speed_dist(v_arr, m_N2, T_val)
    ax1.plot(v_arr, fv * 1e-3, color=color, linewidth=2, label=f'T = {T_val} K')
    
    # Velocidad más probable
    vp = np.sqrt(2 * kB * T_val / m_N2)
    ax1.axvline(x=vp, color=color, linestyle=':', alpha=0.4)

ax1.set_xlabel("Rapidez $v$ [m/s]")
ax1.set_ylabel("$f(v)$ [10$^{-3}$ s/m]")
ax1.set_title("Distribución de Maxwell-Boltzmann (N₂)")
ax1.legend()
ax1.set_xlim(0, 2500)

# Panel 2: Distribución de energías
ax2 = axes[0, 1]
for T_val, color in zip(temperatures, colors_mb):
    E_max = 8 * kB * T_val
    E_arr = np.linspace(0, E_max, 1000)
    fE = maxwell_energy_dist(E_arr, T_val)
    ax2.plot(E_arr / kB, fE * kB, color=color, linewidth=2, label=f'T = {T_val} K')

ax2.set_xlabel("Energía $E/k_B$ [K]")
ax2.set_ylabel("$g(E) \\cdot k_B$")
ax2.set_title("Distribución de energía cinética")
ax2.legend()

# Panel 3: Simulación Monte Carlo - gas 2D
ax3 = axes[1, 0]
np.random.seed(42)
N_particles = 5000
T_sim = 300  # K
sigma_v = np.sqrt(kB * T_sim / m_N2)

# Velocidades Maxwell-Boltzmann (componentes gaussianas)
vx = np.random.normal(0, sigma_v, N_particles)
vy = np.random.normal(0, sigma_v, N_particles)
speeds = np.sqrt(vx**2 + vy**2)

# Histograma vs teórica (2D: distribución Rayleigh)
v_theory = np.linspace(0, 4 * sigma_v, 200)
f_2d = (v_theory / sigma_v**2) * np.exp(-v_theory**2 / (2 * sigma_v**2))

ax3.hist(speeds, bins=60, density=True, alpha=0.6, color='C0', edgecolor='black', 
         linewidth=0.5, label=f'Simulación ({N_particles} partículas)')
ax3.plot(v_theory, f_2d, 'r-', linewidth=2.5, label='Distribución teórica (2D)')
ax3.set_xlabel("Rapidez [m/s]")
ax3.set_ylabel("Densidad de probabilidad")
ax3.set_title(f"Simulación MC de velocidades (T={T_sim}K, 2D)")
ax3.legend()

# Panel 4: Componente vx (debe ser gaussiana)
ax4 = axes[1, 1]
ax4.hist(vx, bins=60, density=True, alpha=0.6, color='C1', edgecolor='black', linewidth=0.5,
         label='Simulación $v_x$')
vx_theory = np.linspace(-3*sigma_v, 3*sigma_v, 200)
gauss = np.exp(-vx_theory**2 / (2 * sigma_v**2)) / (sigma_v * np.sqrt(2 * np.pi))
ax4.plot(vx_theory, gauss, 'r-', linewidth=2.5, label='Gaussiana teórica')
ax4.set_xlabel("$v_x$ [m/s]")
ax4.set_ylabel("Densidad de probabilidad")
ax4.set_title("Componente $v_x$: distribución gaussiana")
ax4.legend()

plt.tight_layout()
plt.show()

# Velocidades características para N2 a 300K
T_ref = 300
vp = np.sqrt(2 * kB * T_ref / m_N2)
v_mean = np.sqrt(8 * kB * T_ref / (np.pi * m_N2))
v_rms = np.sqrt(3 * kB * T_ref / m_N2)

print(f"\n📊 Velocidades de N₂ a T = {T_ref} K:")
print(f"   v_más probable = {vp:.1f} m/s")
print(f"   ⟨v⟩ (media)    = {v_mean:.1f} m/s")
print(f"   v_rms           = {v_rms:.1f} m/s")
print(f"   Relación: v_p : ⟨v⟩ : v_rms = 1 : {v_mean/vp:.3f} : {v_rms/vp:.3f}")

---

## 7. Termodinámica desde la Mecánica Estadística

### 7.1 Las leyes de la termodinámica como teoremas

| Ley | Enunciado termodinámico | Origen estadístico |
|-----|------------------------|-------------------|
| **0ª** | Equilibrio térmico es transitivo | Distribución de Boltzmann con $\beta$ común |
| **1ª** | $dU = \delta Q - \delta W$ | Conservación de $\langle H \rangle$ |
| **2ª** | $\Delta S \geq 0$ | $\Omega$ del sistema combinado crece |
| **3ª** | $S \to 0$ cuando $T \to 0$ | Un solo microestado accesible |

### 7.2 Potenciales termodinámicos

$$\begin{aligned}
U(S,V,N) &\quad \text{(energía interna)} \\
F(T,V,N) &= U - TS \quad \text{(Helmholtz)} \\
H(S,P,N) &= U + PV \quad \text{(entalpía)} \\
G(T,P,N) &= U - TS + PV \quad \text{(Gibbs)}
\end{aligned}$$

### 7.3 Relaciones de Maxwell

De la igualdad de derivadas cruzadas ($\partial^2 F / \partial x \partial y = \partial^2 F / \partial y \partial x$):

$$\left(\frac{\partial T}{\partial V}\right)_S = -\left(\frac{\partial P}{\partial S}\right)_V, \qquad \left(\frac{\partial S}{\partial V}\right)_T = \left(\frac{\partial P}{\partial T}\right)_V$$

$$\left(\frac{\partial T}{\partial P}\right)_S = \left(\frac{\partial V}{\partial S}\right)_P, \qquad \left(\frac{\partial S}{\partial P}\right)_T = -\left(\frac{\partial V}{\partial T}\right)_P$$

In [ ]:
# Derivación simbólica: Del gas ideal → potenciales termodinámicos

print("═" * 60)
print("  GAS IDEAL: Potenciales termodinámicos")
print("═" * 60)

T_s, V_s, N_s, k_s, m_s, h_s = symbols('T V N k_B m h', positive=True)
beta_s = 1 / (k_s * T_s)

# Función de partición Z
# λ = h / sqrt(2πmkT)
lambda_th = h_s / sp.sqrt(2 * sp.pi * m_s * k_s * T_s)
Z_gas = (V_s / lambda_th**3)**N_s / sp.factorial(N_s)

# Energía libre de Helmholtz
# Usamos Stirling: ln(N!) ≈ N ln N - N
F_gas = -k_s * T_s * (N_s * sp.log(V_s / (N_s * lambda_th**3)) + N_s)

print("\n1. Función de partición:")
display(Math(r"Z = \frac{1}{N!}\left(\frac{V}{\lambda^3}\right)^N, \quad \lambda = " + latex(lambda_th)))

print("\n2. Energía libre de Helmholtz (con Stirling):")
F_simplified = simplify(F_gas)
display(Math(r"F = -k_BT\left[N\ln\left(\frac{V}{N\lambda^3}\right) + N\right]"))

# Presión
P_gas = -diff(F_gas, V_s)
P_gas = simplify(P_gas)
print("\n3. Presión P = -∂F/∂V:")
display(Math(r"P = " + latex(P_gas) + r" = \frac{Nk_BT}{V} \quad \checkmark"))

# Entropía (Sackur-Tetrode)
S_gas = -diff(F_gas, T_s)
S_gas = simplify(S_gas)
print("\n4. Entropía S = -∂F/∂T:")
display(Math(r"S = " + latex(S_gas)))

# Energía interna
U_gas = F_gas + T_s * S_gas
U_gas = simplify(U_gas)
print("\n5. Energía interna U = F + TS:")
display(Math(r"U = " + latex(U_gas)))
print(f"\n   → U = (3/2)NkT  (equipartición: 3 grados de libertad traslacionales)")

# Capacidad calorífica
Cv_gas = diff(U_gas, T_s)
print(f"\n6. Capacidad calorífica Cv = ∂U/∂T:")
display(Math(r"C_V = " + latex(simplify(Cv_gas)) + r" = \frac{3}{2}Nk_B"))

print("\n✅ Todo consistente: PV = NkT, U = 3NkT/2, Cv = 3Nk/2")

In [ ]:
# Visualización: Cuadrado termodinámico de Born y potenciales

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Gas ideal - ecuación de estado P(V,T) superficie 3D
ax1 = fig.add_subplot(121, projection='3d')
V_3d = np.linspace(0.5, 5, 50)
T_3d = np.linspace(100, 600, 50)
V_grid, T_grid = np.meshgrid(V_3d, T_3d)

# PV = NkT → P = NkT/V (N=1 mol, en unidades SI simplificadas)
N_mol = 1.0
P_grid = N_mol * R_gas * T_grid / V_grid

# Limitar presión para mejor visualización
P_grid_plot = np.clip(P_grid, 0, 5000)

surf = ax1.plot_surface(V_grid, T_grid, P_grid_plot / 1000, cmap='coolwarm', alpha=0.8)
ax1.set_xlabel('V [L]')
ax1.set_ylabel('T [K]')
ax1.set_zlabel('P [kPa]')
ax1.set_title('Superficie PVT del gas ideal')
ax1.view_init(elev=20, azim=135)

# Panel 2: Isotermas de gas ideal vs Van der Waals
ax2 = axes[1]
V_iso = np.linspace(0.05, 5, 500)

# Van der Waals: (P + a/V²)(V - b) = NkT → P = NkT/(V-b) - a/V²
# Constantes para CO2 (por mol)
a_vdw = 0.3658  # Pa·m⁶/mol²  (simplificado)
b_vdw = 0.00004286  # m³/mol (simplificado)

temps_iso = [200, 300, 400, 500]
colors_iso = plt.cm.hot(np.linspace(0.2, 0.8, len(temps_iso)))

for T_val, color in zip(temps_iso, colors_iso):
    # Gas ideal
    P_ideal = N_mol * R_gas * T_val / V_iso
    ax2.plot(V_iso, P_ideal / 1e3, color=color, linewidth=2, label=f'Ideal T={T_val}K')
    
    # Van der Waals (simplificado)
    P_vdw = R_gas * T_val / (V_iso - 0.04) - 0.4 / V_iso**2
    P_vdw = np.clip(P_vdw, -500, 15000)
    ax2.plot(V_iso, P_vdw / 1e3, '--', color=color, linewidth=1.5, alpha=0.6)

ax2.set_xlabel("V [L]")
ax2.set_ylabel("P [kPa]")
ax2.set_title("Isotermas: Gas ideal (sólido) vs Van der Waals (punteado)")
ax2.legend(fontsize=8)
ax2.set_xlim(0.1, 5)
ax2.set_ylim(0, 15)

plt.tight_layout()
plt.show()

print("📊 El gas ideal es una excelente aproximación para gases diluidos.")
print("   Las desviaciones (Van der Waals) aparecen a alta densidad/baja T.")

---

## 8. Transiciones de Fase: Modelo de Ising 2D

### 8.1 El modelo

El **modelo de Ising** es el modelo más simple de magnetismo y transiciones de fase:

- Red cuadrada de $L \times L$ sitios
- Cada sitio tiene un spin $s_i = \pm 1$
- Hamiltoniano: $H = -J \sum_{\langle i,j \rangle} s_i s_j$ donde la suma es sobre **vecinos más cercanos**
- $J > 0$: ferromagnético (spins prefieren alinearse)

### 8.2 Temperatura crítica (Onsager, 1944)

$$\boxed{T_c = \frac{2J}{k_B \ln(1 + \sqrt{2})} \approx \frac{2.269 J}{k_B}}$$

- $T < T_c$: **fase ordenada** (magnetización espontánea)
- $T > T_c$: **fase desordenada** (paramagnético)
- $T = T_c$: **punto crítico** (fluctuaciones a todas las escalas)

### 8.3 Algoritmo de Metropolis (Monte Carlo)

1. Elegir un spin aleatorio $s_i$
2. Calcular el cambio de energía: $\Delta E = 2J s_i \sum_{j \in \text{vecinos}} s_j$
3. Si $\Delta E \leq 0$: aceptar el flip
4. Si $\Delta E > 0$: aceptar con probabilidad $e^{-\beta \Delta E}$

In [ ]:
# Simulación Monte Carlo del Modelo de Ising 2D (Algoritmo de Metropolis)

def ising_metropolis(L, T_ising, n_steps, J_ising=1.0):
    """Simulación Metropolis del modelo de Ising 2D."""
    beta_ising = 1.0 / T_ising
    spins = np.random.choice([-1, 1], size=(L, L))
    
    magnetization_history = []
    energy_history = []
    
    for step in range(n_steps):
        for _ in range(L * L):  # Un "sweep" = L² intentos
            i = np.random.randint(0, L)
            j = np.random.randint(0, L)
            
            # Suma de vecinos (condiciones periódicas)
            neighbors = (spins[(i+1)%L, j] + spins[(i-1)%L, j] +
                        spins[i, (j+1)%L] + spins[i, (j-1)%L])
            
            dE = 2 * J_ising * spins[i, j] * neighbors
            
            if dE <= 0 or np.random.random() < np.exp(-beta_ising * dE):
                spins[i, j] *= -1
        
        # Medir observables
        magnetization_history.append(np.mean(spins))
        E = -J_ising * np.sum(spins * (np.roll(spins, 1, 0) + np.roll(spins, 1, 1)))
        energy_history.append(E / (L * L))
    
    return spins, np.array(magnetization_history), np.array(energy_history)

# Simular a diferentes temperaturas
L_ising = 30
n_steps_ising = 200
Tc_ising = 2.0 / np.log(1 + np.sqrt(2))  # ≈ 2.269

print(f"Simulando modelo de Ising {L_ising}×{L_ising}...")
print(f"Temperatura crítica: Tc = {Tc_ising:.4f} J/kB")

T_values_ising = [1.0, 2.0, Tc_ising, 2.5, 3.5]
results_ising = {}

for T_is in T_values_ising:
    spins_final, mag_hist, E_hist = ising_metropolis(L_ising, T_is, n_steps_ising)
    results_ising[T_is] = (spins_final, mag_hist, E_hist)
    print(f"   T = {T_is:.3f}: ⟨m⟩ = {np.mean(np.abs(mag_hist[-50:])):.3f}")

# Visualización
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Fila 1: Configuraciones de spins
for idx, T_is in enumerate([1.0, Tc_ising, 3.5]):
    ax = axes[0, idx]
    spins_show = results_ising[T_is][0]
    ax.imshow(spins_show, cmap='RdBu', interpolation='nearest', vmin=-1, vmax=1)
    phase = "Ordenada" if T_is < Tc_ising else ("Crítica" if abs(T_is - Tc_ising) < 0.01 else "Desordenada")
    ax.set_title(f"T = {T_is:.2f} J/kB ({phase})")
    ax.set_xticks([])
    ax.set_yticks([])

# Fila 2, Panel 1: Magnetización vs tiempo (termalización)
ax_m = axes[1, 0]
for T_is in [1.0, 2.0, Tc_ising, 3.5]:
    mag = results_ising[T_is][1]
    ax_m.plot(np.abs(mag), linewidth=1, label=f'T={T_is:.2f}', alpha=0.8)
ax_m.set_xlabel("Paso MC")
ax_m.set_ylabel("$|m|$")
ax_m.set_title("Magnetización vs pasos Monte Carlo")
ax_m.legend(fontsize=8)

# Fila 2, Panel 2: Magnetización promedio vs T
ax_mt = axes[1, 1]
T_scan = np.linspace(1.0, 4.0, 20)
mag_avg = []
mag_std = []

for T_sc in T_scan:
    _, mag_h, _ = ising_metropolis(L_ising, T_sc, 150)
    m_eq = np.abs(mag_h[-50:])  # últimos 50 pasos (equilibrio)
    mag_avg.append(np.mean(m_eq))
    mag_std.append(np.std(m_eq))

ax_mt.errorbar(T_scan, mag_avg, yerr=mag_std, fmt='o-', color='C0', 
               markersize=5, capsize=3, linewidth=1.5)
ax_mt.axvline(x=Tc_ising, color='red', linestyle='--', linewidth=2, 
              label=f'$T_c$ = {Tc_ising:.3f}')
ax_mt.set_xlabel("$T$ [J/kB]")
ax_mt.set_ylabel("$\\langle |m| \\rangle$")
ax_mt.set_title("Transición de fase: magnetización vs T")
ax_mt.legend()

# Fila 2, Panel 3: Energía vs T
ax_et = axes[1, 2]
E_avg = []
for T_sc in T_scan:
    _, _, E_h = ising_metropolis(L_ising, T_sc, 150)
    E_avg.append(np.mean(E_h[-50:]))

ax_et.plot(T_scan, E_avg, 'o-', color='C1', markersize=5, linewidth=1.5)
ax_et.axvline(x=Tc_ising, color='red', linestyle='--', linewidth=2, label=f'$T_c$')
ax_et.set_xlabel("$T$ [J/kB]")
ax_et.set_ylabel("$\\langle E \\rangle / N$")
ax_et.set_title("Energía por sitio vs temperatura")
ax_et.legend()

plt.tight_layout()
plt.show()

print("\n🧲 Transición de fase de segundo orden:")
print(f"   T < Tc ({Tc_ising:.3f}): spins ordenados, |m| → 1")
print(f"   T > Tc: spins desordenados, |m| → 0")
print(f"   T = Tc: fluctuaciones críticas, dominios a todas las escalas")

---

## 9. Teorema de Equipartición y Límites de la Física Clásica

### 9.1 Teorema de Equipartición

> **Teorema:** En equilibrio térmico a temperatura $T$, cada grado de libertad cuadrático en el Hamiltoniano contribuye $\frac{1}{2}k_BT$ a la energía media.

Si $H = \sum_i \alpha_i q_i^2 + \sum_j \beta_j p_j^2$, entonces:

$$\langle E \rangle = \frac{f}{2} k_B T$$

donde $f$ es el número de términos cuadráticos.

### 9.2 Aplicaciones

| Sistema | Grados cuadráticos | $C_V$ predicho |
|---------|-------------------|---------------|
| Gas monoatómico | 3 (traslación) | $\frac{3}{2}Nk_B$ |
| Gas diatómico | 5 (3 trasl. + 2 rot.) | $\frac{5}{2}Nk_B$ |
| Gas diatómico (con vibración) | 7 (+ 2 vibr.) | $\frac{7}{2}Nk_B$ |
| Sólido cristalino | 6 (3 cin. + 3 pot.) | $3Nk_B$ (ley de Dulong-Petit) |

### 9.3 Los fracasos de la física clásica

La mecánica estadística clásica predice:

1. **Catástrofe ultravioleta:** La radiación del cuerpo negro tendría energía infinita ($\langle E \rangle = k_BT$ por modo)
2. **Calor específico de sólidos:** $C_V = 3Nk_B$ a **toda** temperatura (experimentalmente, $C_V \to 0$ cuando $T \to 0$)
3. **Calor específico de gases diatómicos:** $C_V = \frac{7}{2}Nk_B$ (experimentalmente se "congelan" grados de libertad)

> Estos fracasos motivaron la mecánica cuántica: **la energía está cuantizada**, y a baja $T$ los modos de alta frecuencia se "congelan".

In [ ]:
# Los límites de la física clásica: Equipartición vs Realidad

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Cv de gas diatómico vs T (escalones cuánticos)
ax1 = axes[0]
T_diat = np.logspace(1, 4, 500)  # 10 K a 10000 K

# Modelo cuántico simplificado: rotación + vibración
Theta_rot = 85.4    # K (temperatura rotacional de H2)
Theta_vib = 6215    # K (temperatura vibracional de H2)

# Contribución rotacional (clásica: kT >> Θ_rot)
x_rot = Theta_rot / T_diat
Cv_rot = x_rot**2 * np.exp(x_rot) / (np.exp(x_rot) - 1)**2  # por molécula
# Suavizar la transición
Cv_rot = np.where(T_diat > 3 * Theta_rot, 1.0, Cv_rot * 2)

# Contribución vibracional (cuántica)
x_vib = Theta_vib / T_diat
Cv_vib = x_vib**2 * np.exp(x_vib) / (np.exp(x_vib) - 1)**2

# Cv total / (NkB): 3/2 (trasl) + Cv_rot + Cv_vib
Cv_total = 1.5 + Cv_rot + Cv_vib

ax1.semilogx(T_diat, Cv_total, 'C0', linewidth=2.5, label='Modelo cuántico')
ax1.axhline(y=3.5, color='C1', linestyle='--', linewidth=2, label='Clásico: $\\frac{7}{2}$ (equip.)')
ax1.axhline(y=2.5, color='C2', linestyle=':', linewidth=1.5, label='$\\frac{5}{2}$ (trasl. + rot.)')
ax1.axhline(y=1.5, color='C3', linestyle=':', linewidth=1.5, label='$\\frac{3}{2}$ (solo trasl.)')

ax1.set_xlabel("T [K]")
ax1.set_ylabel("$C_V / Nk_B$")
ax1.set_title("$C_V$ de gas diatómico (H₂)\nEscalones cuánticos vs equipartición")
ax1.legend(fontsize=8)
ax1.set_ylim(1, 4)
ax1.annotate('Rotación\nse activa', xy=(200, 2.5), fontsize=9, ha='center')
ax1.annotate('Vibración\nse activa', xy=(3000, 3.2), fontsize=9, ha='center')

# Panel 2: Modelo de Einstein para sólidos (Cv → 0 a baja T)
ax2 = axes[1]
T_solid = np.linspace(1, 1000, 500)
Theta_E = 200  # Temperatura de Einstein (K)

x_E = Theta_E / T_solid
Cv_einstein = 3 * x_E**2 * np.exp(x_E) / (np.exp(x_E) - 1)**2

ax2.plot(T_solid / Theta_E, Cv_einstein, 'C0', linewidth=2.5, label='Modelo de Einstein')
ax2.axhline(y=3, color='C1', linestyle='--', linewidth=2, label='Dulong-Petit: $3Nk_B$')
ax2.set_xlabel("$T / \\Theta_E$")
ax2.set_ylabel("$C_V / Nk_B$")
ax2.set_title(f"Calor específico de sólidos\n$\\Theta_E$ = {Theta_E} K")
ax2.legend()
ax2.set_xlim(0, 5)
ax2.annotate('$C_V \\to 0$\n(3ª ley)', xy=(0.3, 0.3), fontsize=10, color='red')

# Panel 3: Catástrofe ultravioleta - Rayleigh-Jeans vs Planck
ax3 = axes[2]
T_bb = 5000  # K (similar al Sol)
nu = np.linspace(0.01e14, 3e15, 1000)  # frecuencia en Hz
h_planck = 6.626e-34
c_light = 3e8

# Rayleigh-Jeans (clásico): u(ν) = 8πν²kT/c³
u_RJ = 8 * np.pi * nu**2 * kB * T_bb / c_light**3

# Planck (cuántico): u(ν) = 8πhν³/c³ · 1/(e^{hν/kT} - 1)
x_planck = h_planck * nu / (kB * T_bb)
u_planck = 8 * np.pi * h_planck * nu**3 / c_light**3 / (np.exp(x_planck) - 1)

# Normalizar para visualización
norm = np.max(u_planck)
ax3.plot(nu / 1e14, u_planck / norm, 'C0', linewidth=2.5, label='Planck (cuántico)')
ax3.plot(nu / 1e14, np.clip(u_RJ / norm, 0, 5), 'C1--', linewidth=2, label='Rayleigh-Jeans (clásico)')
ax3.fill_between(nu / 1e14, u_planck / norm, np.clip(u_RJ / norm, 0, 5), 
                  where=u_RJ/norm > u_planck/norm, alpha=0.2, color='red',
                  label='Catástrofe UV')
ax3.set_xlabel("Frecuencia [$10^{14}$ Hz]")
ax3.set_ylabel("Densidad espectral (normalizada)")
ax3.set_title(f"Radiación del cuerpo negro (T = {T_bb} K)\nCatástrofe ultravioleta")
ax3.legend(fontsize=8)
ax3.set_xlim(0, 30)
ax3.set_ylim(0, 3)

plt.tight_layout()
plt.show()

print("⚠️  FRACASOS DE LA MECÁNICA ESTADÍSTICA CLÁSICA:")
print("   1. Catástrofe UV: la densidad clásica diverge a alta frecuencia")
print("   2. Cv de sólidos: clásicamente Cv = 3NkB a toda T (viola 3ª ley)")
print("   3. Cv de gases: clásicamente todos los grados están activos")
print("\n→ La solución: CUANTIZACIÓN de la energía (Planck 1900, Einstein 1907)")

---

## 10. Ejercicios Propuestos y Resueltos

### Nivel 1: Fundamentos

**Ejercicio 1.1** — Calcule la multiplicidad $\Omega$ de un sistema de $N = 4$ osciladores armónicos cuánticos con energía total $E = 6\hbar\omega$. (Hint: usar la fórmula de "estrellas y barras".)

**Ejercicio 1.2** — Demuestre que la entropía de Boltzmann es aditiva: si dos sistemas independientes tienen multiplicidades $\Omega_1$ y $\Omega_2$, entonces $S_{total} = S_1 + S_2$.

**Ejercicio 1.3** — Para un gas ideal 2D (partículas en una superficie de área $A$), derive la ecuación de estado $PA = Nk_BT$.

### Nivel 2: Ensamble Canónico

**Ejercicio 2.1** — Calcule la función de partición, energía media y capacidad calorífica de un sistema con 3 niveles de energía: $E = 0, \epsilon, 3\epsilon$.

**Ejercicio 2.2** — Demuestre que $\sigma_E^2 = k_BT^2 C_V$ partiendo de $\langle E^2 \rangle - \langle E \rangle^2 = \frac{\partial^2 \ln Z}{\partial \beta^2}$.

**Ejercicio 2.3** — Para un oscilador armónico clásico 1D ($H = p^2/2m + kx^2/2$), calcule $Z$, $\langle E \rangle$ y verifique el teorema de equipartición.

### Nivel 3: Maxwell-Boltzmann

**Ejercicio 3.1** — Derive $\langle v \rangle$ y $\langle v^2 \rangle$ integrando la distribución de Maxwell-Boltzmann.

**Ejercicio 3.2** — Calcule la tasa de efusión molecular (partículas por segundo que escapan por un agujero de área $A$) para un gas ideal.

### Nivel 4: Avanzados

**Ejercicio 4.1** — Para el modelo de Ising 1D con $N$ spins y condiciones periódicas, calcule $Z$ exactamente usando la **matriz de transferencia**.

**Ejercicio 4.2** — Derive la ecuación de Van der Waals a partir de la función de partición de un gas con potencial de interacción de par.

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN Ejercicio 1.1: Multiplicidad de osciladores
# ═══════════════════════════════════════════════════
from math import comb

print("═" * 60)
print("  Ejercicio 1.1: Multiplicidad de N osciladores")
print("═" * 60)

display(Markdown(r"""
**N** osciladores armónicos compartiendo **q** cuantos de energía.

La multiplicidad es el número de formas de distribuir $q$ objetos idénticos en $N$ cajas:

$$\Omega(N, q) = \binom{q + N - 1}{q} = \frac{(q + N - 1)!}{q!(N-1)!}$$

Para $N = 4$, $q = 6$:
"""))

N_osc, q_osc = 4, 6
omega_osc = comb(q_osc + N_osc - 1, q_osc)
S_osc = np.log(omega_osc)

print(f"   Ω({N_osc}, {q_osc}) = C({q_osc + N_osc - 1}, {q_osc}) = C(9, 6) = {omega_osc}")
print(f"   S/kB = ln({omega_osc}) = {S_osc:.4f}")

# Visualización: Ω vs q para N fijo
q_range = np.arange(0, 30)
omega_vs_q = np.array([comb(q + N_osc - 1, q) for q in q_range])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.semilogy(q_range, omega_vs_q, 'C0o-', markersize=5, linewidth=1.5)
ax1.set_xlabel("$q$ (cuantos de energía)")
ax1.set_ylabel("$\\Omega(N=4, q)$")
ax1.set_title(f"Multiplicidad vs energía (N={N_osc} osciladores)")
ax1.plot(q_osc, omega_osc, 'r*', markersize=15, label=f'q={q_osc}: Ω={omega_osc}')
ax1.legend()

# Dos subsistemas en contacto: maximización de Ω_total
ax2 = axes[1]
N_A, N_B = 4, 4
q_total = 10
q_A_range = np.arange(0, q_total + 1)
q_B_range = q_total - q_A_range

omega_A = np.array([comb(q + N_A - 1, q) for q in q_A_range])
omega_B = np.array([comb(q + N_B - 1, q) for q in q_B_range])
omega_total = omega_A * omega_B

ax2.bar(q_A_range, omega_total, color='C1', edgecolor='black', alpha=0.7)
ax2.set_xlabel("$q_A$ (energía en subsistema A)")
ax2.set_ylabel("$\\Omega_A \\cdot \\Omega_B$")
ax2.set_title(f"Equilibrio: Ω_total se maximiza\n$N_A={N_A}, N_B={N_B}, q_{{total}}={q_total}$")

q_eq = q_A_range[np.argmax(omega_total)]
ax2.axvline(x=q_eq, color='red', linestyle='--', linewidth=2, label=f'Equilibrio: $q_A$={q_eq}')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\n📊 El equilibrio ocurre cuando q_A = {q_eq} (energía equipartida)")
print(f"   Esto maximiza Ω_total → maximiza S_total (2ª ley)")

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN Ejercicio 2.1: Sistema de 3 niveles
# ═══════════════════════════════════════════════════

print("═" * 60)
print("  Ejercicio 2.1: Sistema de 3 niveles (E = 0, ε, 3ε)")
print("═" * 60)

# Simbólico
beta_sym = symbols('beta', positive=True)
eps_sym = symbols('epsilon', positive=True)

Z_3lvl_sym = 1 + sp.exp(-beta_sym * eps_sym) + sp.exp(-3 * beta_sym * eps_sym)
print("\n1. Función de partición:")
display(Math(r"Z = 1 + e^{-\beta\varepsilon} + e^{-3\beta\varepsilon} = " + latex(Z_3lvl_sym)))

E_avg_3lvl = -diff(sp.log(Z_3lvl_sym), beta_sym)
E_avg_3lvl = simplify(E_avg_3lvl)
print("\n2. Energía media ⟨E⟩ = -∂ln(Z)/∂β:")
display(Math(r"\langle E \rangle = " + latex(E_avg_3lvl)))

# Numérico
beta_num = np.linspace(0.01, 5, 500)
Z_3lvl_num = 1 + np.exp(-beta_num) + np.exp(-3 * beta_num)  # ε = 1
E_3lvl_num = (np.exp(-beta_num) + 3 * np.exp(-3 * beta_num)) / Z_3lvl_num
Cv_3lvl_num = np.gradient(E_3lvl_num, -1.0/beta_num)  # dE/dT, T = 1/β

# Probabilidades
P0 = 1 / Z_3lvl_num
P1 = np.exp(-beta_num) / Z_3lvl_num
P3 = np.exp(-3 * beta_num) / Z_3lvl_num

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Probabilidades de ocupación
ax1 = axes[0]
T_3lvl = 1.0 / beta_num
ax1.plot(T_3lvl, P0, 'C0', linewidth=2, label='$P(E=0)$')
ax1.plot(T_3lvl, P1, 'C1', linewidth=2, label='$P(E=\\varepsilon)$')
ax1.plot(T_3lvl, P3, 'C2', linewidth=2, label='$P(E=3\\varepsilon)$')
ax1.axhline(y=1/3, color='gray', linestyle='--', alpha=0.5, label='1/3 (equiprobable)')
ax1.set_xlabel("$k_BT / \\varepsilon$")
ax1.set_ylabel("Probabilidad")
ax1.set_title("Probabilidades de ocupación")
ax1.legend()
ax1.set_xlim(0, 5)

# Panel 2: Energía media
ax2 = axes[1]
ax2.plot(T_3lvl, E_3lvl_num, 'C0', linewidth=2.5)
ax2.axhline(y=4/3, color='gray', linestyle='--', alpha=0.5, label='$T\\to\\infty$: $\\langle E \\rangle = 4\\varepsilon/3$')
ax2.axhline(y=0, color='C1', linestyle='--', alpha=0.5, label='$T\\to 0$: $\\langle E \\rangle = 0$')
ax2.set_xlabel("$k_BT / \\varepsilon$")
ax2.set_ylabel("$\\langle E \\rangle / \\varepsilon$")
ax2.set_title("Energía media")
ax2.legend()
ax2.set_xlim(0, 5)

# Panel 3: Capacidad calorífica
ax3 = axes[2]
# Calcular Cv analíticamente
dE_dbeta = np.gradient(E_3lvl_num, beta_num)
Cv_analytic = -beta_num**2 * dE_dbeta

ax3.plot(T_3lvl, Cv_analytic, 'C1', linewidth=2.5)
ax3.set_xlabel("$k_BT / \\varepsilon$")
ax3.set_ylabel("$C_V / k_B$")
ax3.set_title("Capacidad calorífica")
ax3.set_xlim(0, 5)
ax3.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f"\n📊 Resultados del sistema de 3 niveles:")
print(f"   T → 0: solo el estado base está ocupado (P₀ → 1)")
print(f"   T → ∞: todos equiprobables (P = 1/3), ⟨E⟩ = 4ε/3")

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN Ejercicio 4.1: Modelo de Ising 1D - Matriz de Transferencia
# ═══════════════════════════════════════════════════

print("═" * 60)
print("  Ejercicio 4.1: Ising 1D con Matriz de Transferencia")
print("═" * 60)

display(Markdown(r"""
**Hamiltoniano:** $H = -J\sum_{i=1}^{N} s_i s_{i+1}$ (condiciones periódicas)

**Matriz de transferencia:**
$$T = \begin{pmatrix} e^{\beta J} & e^{-\beta J} \\ e^{-\beta J} & e^{\beta J} \end{pmatrix}$$

**Eigenvalores:** $\lambda_{\pm} = e^{\beta J} \pm e^{-\beta J}$, es decir $\lambda_+ = 2\cosh(\beta J)$, $\lambda_- = 2\sinh(\beta J)$

**Función de partición:** $Z = \text{Tr}(T^N) = \lambda_+^N + \lambda_-^N$

Para $N \to \infty$: $Z \approx \lambda_+^N = [2\cosh(\beta J)]^N$
"""))

# Verificación numérica: Z exacta vs fuerza bruta para N pequeño
def ising_1d_exact_Z(N_ising, beta_J):
    """Z exacta usando matriz de transferencia."""
    lp = 2 * np.cosh(beta_J)
    lm = 2 * np.sinh(beta_J)
    return lp**N_ising + lm**N_ising

def ising_1d_brute_force_Z(N_ising, beta_J):
    """Z por suma explícita sobre todos los 2^N estados."""
    Z_bf = 0
    for state in range(2**N_ising):
        spins_bf = np.array([(state >> i) & 1 for i in range(N_ising)]) * 2 - 1
        E = -sum(spins_bf[i] * spins_bf[(i+1) % N_ising] for i in range(N_ising))
        Z_bf += np.exp(-beta_J * E)  # β*J ya está incorporado en beta_J
    return Z_bf

# Comparar para N = 8
N_test = 8
bJ_range = np.linspace(0.01, 3, 50)

print(f"\nVerificación para N = {N_test}:")
print(f"{'βJ':>6} {'Z (transferencia)':>20} {'Z (fuerza bruta)':>20} {'Error rel':>12}")
print("─" * 62)

for bJ in [0.1, 0.5, 1.0, 2.0]:
    Z_tm = ising_1d_exact_Z(N_test, bJ)
    Z_bf = ising_1d_brute_force_Z(N_test, bJ)
    err = abs(Z_tm - Z_bf) / Z_bf
    print(f"{bJ:>6.1f} {Z_tm:>20.6f} {Z_bf:>20.6f} {err:>12.2e}")

# Termodinámica exacta del Ising 1D
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

bJ_plot = np.linspace(0.01, 4, 500)
T_ising1d = 1.0 / bJ_plot  # T en unidades de J/kB

# Energía libre: f = -T ln(2cosh(βJ)) = -(1/β) ln(2cosh(βJ))
f_ising = -(1.0 / bJ_plot) * np.log(2 * np.cosh(bJ_plot))

# Energía: <E>/N = -J tanh(βJ)
E_ising1d = -np.tanh(bJ_plot)

# Capacidad calorífica: Cv/NkB = (βJ)² / cosh²(βJ)
Cv_ising1d = bJ_plot**2 / np.cosh(bJ_plot)**2

# Entropía: S/NkB = βJ·tanh(βJ) - ln(2cosh(βJ)) + ... = ln(2cosh(βJ)) - βJ·tanh(βJ)
S_ising1d = np.log(2 * np.cosh(bJ_plot)) - bJ_plot * np.tanh(bJ_plot)

ax1 = axes[0]
ax1.plot(T_ising1d, E_ising1d, 'C0', linewidth=2.5)
ax1.set_xlabel("$k_BT / J$")
ax1.set_ylabel("$\\langle E \\rangle / NJ$")
ax1.set_title("Energía por sitio (Ising 1D)")
ax1.set_xlim(0, 5)

ax2 = axes[1]
ax2.plot(T_ising1d, Cv_ising1d, 'C1', linewidth=2.5)
ax2.set_xlabel("$k_BT / J$")
ax2.set_ylabel("$C_V / Nk_B$")
ax2.set_title("Capacidad calorífica (sin transición de fase)")
ax2.set_xlim(0, 5)

ax3 = axes[2]
ax3.plot(T_ising1d, S_ising1d, 'C2', linewidth=2.5)
ax3.axhline(y=np.log(2), color='gray', linestyle='--', alpha=0.5, label='$\\ln 2$')
ax3.set_xlabel("$k_BT / J$")
ax3.set_ylabel("$S / Nk_B$")
ax3.set_title("Entropía")
ax3.legend()
ax3.set_xlim(0, 5)

plt.tight_layout()
plt.show()

print("\n📊 Resultado clave del Ising 1D:")
print("   NO hay transición de fase a T > 0 (Cv es suave)")
print("   Se necesitan d ≥ 2 dimensiones para una transición de fase")

---

## Resumen y Mapa Conceptual

```
              TERMODINÁMICA ESTADÍSTICA CLÁSICA
                         │
         ┌───────────────┼───────────────┐
         │               │               │
    ENSAMBLE         ENSAMBLE        ENSAMBLE GRAN
    MICROCANÓNICO    CANÓNICO        CANÓNICO
    (N,V,E fijos)    (N,V,T fijos)   (μ,V,T fijos)
         │               │               │
    Ω(E,V,N)         Z(N,V,T)        Z(μ,V,T)
    S = kB ln Ω      F = -kT ln Z    Φ = -kT ln Z
         │               │               │
         └───────┬───────┘               │
                 │                       │
          GAS IDEAL CLÁSICO         SISTEMAS CON
          PV = NkBT                 INTERACCIÓN
          Maxwell-Boltzmann         Ising, Van der Waals
                 │
         ┌───────┼───────┐
         │       │       │
    EQUIPAR-  POTENCIA-  RELACIONES
    TICIÓN    LES TERM.  DE MAXWELL
    ⟨E⟩=f/2kT F,G,H,U   (∂S/∂V)=(∂P/∂T)
         │
    LÍMITES CLÁSICOS
    Catástrofe UV
    Cv → 0 (T→0)
         │
    → MECÁNICA CUÁNTICA
```

### Texto de referencia

1. **Pathria, R.K.** — *Statistical Mechanics* (3ª ed.) — Referencia completa y rigurosa
2. **Reif, F.** — *Fundamentals of Statistical and Thermal Physics* — Excelente pedagógicamente
3. **Schroeder, D.V.** — *An Introduction to Thermal Physics* — Ideal para comenzar
4. **Huang, K.** — *Statistical Mechanics* (2ª ed.) — Enfoque formal

---

*Notebook de Termodinámica Estadística Clásica. Ejecutar todas las celdas secuencialmente. La simulación de Ising 2D puede tomar ~1 minuto.*